# 第3节：数字音频基础 - 动手实验

本notebook包含第3节课的所有实验代码，请按顺序执行每个单元。

## 1. 导入必要的库

In [ ]:
import soundfile as sf
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
import os

print("✅ 所有库导入成功！")
print(f"NumPy版本: {np.__version__}")

## 2. 读取音频文件并分析基本信息

In [ ]:
# 音频文件路径
audio_path = '../assets/audio/test_stereo.wav'

# 检查文件是否存在
if not os.path.exists(audio_path):
    print("⚠️  测试音频不存在，请先下载：")
    print("mkdir -p ../assets/audio")
    print("wget https://filesamples.com/samples/audio/wav/sample1.wav -O ../assets/audio/test_stereo.wav")
else:
    # 读取音频文件
    data, sample_rate = sf.read(audio_path)
    
    print(f"✅ 音频读取成功！")
    print(f"=== 音频基本信息 ===")
    print(f"音频文件: {audio_path}")
    print(f"采样率: {sample_rate} Hz")
    print(f"声道数: {data.shape[1] if len(data.shape) > 1 else 1}")
    print(f"总采样点数: {len(data):,}")
    print(f"音频时长: {len(data) / sample_rate:.2f} 秒")
    print(f"数据类型: {data.dtype}")
    print(f"数据范围: [{data.min():.4f}, {data.max():.4f}]")
    
    # 计算文件大小
    if len(data.shape) > 1:
        file_size_bytes = len(data) * data.itemsize * data.shape[1]
    else:
        file_size_bytes = len(data) * data.itemsize
    print(f"原始PCM数据大小: {file_size_bytes:,} 字节 ({file_size_bytes/1024/1024:.2f} MB)")

## 3. 绘制音频波形图

## 6. 录制音频并分析（可选，需要麦克风）

In [ ]:
import pyaudio
import wave

def record_audio(duration=3, sample_rate=44100, channels=1):
    """录制音频"""
    CHUNK = 1024
    FORMAT = pyaudio.paInt16
    CHANNELS = channels
    RATE = sample_rate
    
    p = pyaudio.PyAudio()
    
    print(f"开始录制  {duration} 秒音频...")
    print(f"参数: { sample_rate} Hz,  {channels} 声道, 16位深度")
    
    stream = p.open(format=FORMAT,
                    channels=CHANNELS,
                    rate=RATE,
                    input=True,
                    frames_per_buffer=CHUNK)
    
    frames = []
    
    for i in range(0, int(RATE / CHUNK * duration)):
        data = stream.read(CHUNK)
        frames.append(data)
    
    print("录制完成!")
    
    stream.stop_stream()
    stream.close()
    p.terminate()
    
    # 转换为numpy数组
    audio_data = np.frombuffer(b''.join(frames), dtype=np.int16)
    
    # 归一化到[-1, 1]
    audio_data = audio_data.astype(np.float32) / np.iinfo(np.int16).max
    
    return audio_data, RATE

# 录制音频（需要麦克风）
try:
    recorded_data, record_rate = record_audio(duration=3)
    
    # 绘制录制的音频波形
    plt.figure(figsize=(12, 4))
    time_axis = np.linspace(0, 3, len(recorded_data))
    plt.plot(time_axis, recorded_data, alpha=0.7)
    plt.title('录制的音频波形', fontsize=14)
    plt.xlabel('时间 (秒)')
    plt.ylabel('振幅')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"=== 录制音频分析 ===")
    print(f"采样点数: { len(recorded_data) }")
    print(f"最大值: { recorded_data.max():.4f }, 最小值: { recorded_data.min():.4f }")
    print(f"平均音量: { np.mean(np.abs(recorded_data)):.4f }")
    
except Exception as e:
    print(f"麦克风录制失败: { e }")
    print("请确保已安装pyaudio并连接麦克风")
    print("pip install pyaudio")

In [ ]:
# 创建时间轴
duration = len(data) / sample_rate
time_axis = np.linspace(0, duration, len(data))

# 绘制波形图
plt.figure(figsize=(12, 6))

if len(data.shape) > 1:
    # 立体声
    plt.subplot(2, 1, 1)
    plt.plot(time_axis, data[:, 0], alpha=0.7)
    plt.title('左声道波形图', fontsize=14)
    plt.xlabel('时间 (秒)')
    plt.ylabel('振幅')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 1, 2)
    plt.plot(time_axis, data[:, 1], alpha=0.7, color='orange')
    plt.title('右声道波形图', fontsize=14)
    plt.xlabel('时间 (秒)')
    plt.ylabel('振幅')
    plt.grid(True, alpha=0.3)
else:
    # 单声道
    plt.plot(time_axis, data, alpha=0.7)
    plt.title('单声道波形图', fontsize=14)
    plt.xlabel('时间 (秒)')
    plt.ylabel('振幅')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 显示波形统计信息
print(f"=== 波形统计信息 ===")
if len(data.shape) > 1:
    print(f"左声道 - 最大值: {data[:, 0].max():.4f}, 最小值: {data[:, 0].min():.4f}, 平均值: {data[:, 0].mean():.4f}")
    print(f"右声道 - 最大值: {data[:, 1].max():.4f}, 最小值: {data[:, 1].min():.4f}, 平均值: {data[:, 1].mean():.4f}")
else:
    print(f"最大值: {data.max():.4f}, 最小值: {data.min():.4f}, 平均值: {data.mean():.4f}")

## 4. 绘制音频频谱图

In [ ]:
# 计算频谱的辅助函数
def compute_spectrum(audio_data, sample_rate):
    """计算音频信号的频谱"""
    # 使用汉明窗减少频谱泄漏
    window = signal.windows.hamming(len(audio_data))
    windowed_data = audio_data * window
    
    # 计算FFT
    n = len(windowed_data)
    yf = fft(windowed_data)
    xf = fftfreq(n, 1 / sample_rate)
    
    # 只取正频率部分
    half_n = n // 2
    frequencies = xf[:half_n]
    magnitude = np.abs(yf[:half_n]) / n * 2
    
    return frequencies, magnitude

# 绘制频谱图
plt.figure(figsize=(12, 8))

if len(data.shape) > 1:
    # 左声道频谱
    freqs_left, mag_left = compute_spectrum(data[:, 0], sample_rate)
    
    plt.subplot(2, 1, 1)
    plt.semilogx(freqs_left[1:], 20 * np.log10(mag_left[1:] + 1e-10), alpha=0.7)
    plt.title('左声道频谱图', fontsize=14)
    plt.xlabel('频率 (Hz)')
    plt.ylabel('幅度 (dB)')
    plt.grid(True, alpha=0.3)
    plt.xlim(20, sample_rate/2)
    
    # 右声道频谱
    freqs_right, mag_right = compute_spectrum(data[:, 1], sample_rate)
    
    plt.subplot(2, 1, 2)
    plt.semilogx(freqs_right[1:], 20 * np.log10(mag_right[1:] + 1e-10), alpha=0.7, color='orange')
    plt.title('右声道频谱图', fontsize=14)
    plt.xlabel('频率 (Hz)')
    plt.ylabel('幅度 (dB)')
    plt.grid(True, alpha=0.3)
    plt.xlim(20, sample_rate/2)
else:
    # 单声道频谱
    freqs, mag = compute_spectrum(data, sample_rate)
    
    plt.semilogx(freqs[1:], 20 * np.log10(mag[1:] + 1e-10), alpha=0.7)
    plt.title('单声道频谱图', fontsize=14)
    plt.xlabel('频率 (Hz)')
    plt.ylabel('幅度 (dB)')
    plt.grid(True, alpha=0.3)
    plt.xlim(20, sample_rate/2)

plt.tight_layout()
plt.show()

# 显示频谱统计信息
print(f"=== 频谱统计信息 ===")
if len(data.shape) > 1:
    peak_freq_left = freqs_left[np.argmax(mag_left)]
    peak_freq_right = freqs_right[np.argmax(mag_right)]
    print(f"左声道 - 峰值频率: {peak_freq_left:.1f} Hz, 峰值幅度: {20*np.log10(mag_left.max()):.1f} dB")
    print(f"右声道 - 峰值频率: {peak_freq_right:.1f} Hz, 峰值幅度: {20*np.log10(mag_right.max()):.1f} dB")
else:
    peak_freq = freqs[np.argmax(mag)]
    print(f"峰值频率: {peak_freq:.1f} Hz, 峰值幅度: {20*np.log10(mag.max()):.1f} dB")

## 5. 对比不同采样率的音频

In [ ]:
def resample_audio(audio_data, original_rate, target_rate):
    """重采样音频到目标采样率"""
    # 计算重采样比例
    ratio = target_rate / original_rate
    
    # 重采样
    resampled = signal.resample(audio_data, int(len(audio_data) * ratio))
    
    return resampled, target_rate

# 创建不同采样率的版本
rates_to_test = [8000, 16000, 22050, 44100]

plt.figure(figsize=(15, 10))

for i, target_rate in enumerate(rates_to_test, 1):
    # 重采样（只取左声道）
    if len(data.shape) > 1:
        channel_data = data[:, 0]
    else:
        channel_data = data
    
    resampled_data, new_rate = resample_audio(channel_data, sample_rate, target_rate)
    
    # 计算频谱
    freqs, mag = compute_spectrum(resampled_data, new_rate)
    
    # 绘制频谱对比
    plt.subplot(2, 2, i)
    plt.semilogx(freqs[1:], 20 * np.log10(mag[1:] + 1e-10), alpha=0.7)
    plt.title(f'{target_rate} Hz 采样率', fontsize=12)
    plt.xlabel('频率 (Hz)')
    plt.ylabel('幅度 (dB)')
    plt.grid(True, alpha=0.3)
    plt.xlim(20, target_rate/2)
    plt.axvline(x=target_rate/2, color='red', linestyle='--', alpha=0.5, label='奈奎斯特频率')
    plt.legend()

plt.tight_layout()
plt.show()

print(f"=== 采样率对比分析 ===")
for target_rate in rates_to_test:
    nyquist = target_rate / 2
    label = ''
    if target_rate == 8000:
        label = '(电话质量)'
    elif target_rate == 44100:
        label = '(CD质量)'
    print(f"{target_rate}Hz采样率: 最高可记录频率 = {nyquist:.1f}Hz {label}")

## 🎯 课后挑战：尝试以下任务

### 挑战1：音频音量分析
计算音频的平均音量（RMS）和峰值音量，并显示音量随时间的变化曲线。

### 挑战2：手动实现RMS音量计算
计算音频的均方根（RMS）音量：
```
RMS = sqrt(mean(square(audio_data)))
```

## 📚 总结

恭喜完成第3节课的学习！你已经掌握了：

- ✅ 数字音频的核心概念：采样率、位深度、声道数
- ✅ PCM音频格式的存储方式和计算
- ✅ 音频波形图的绘制和时域分析
- ✅ 音频频谱图的绘制和频域分析
- ✅ 不同采样率对音频质量的影响
- ✅ 音频数据的统计分析

**下节课预告：** 第4节 - 音视频容器格式与编解码
- 学习容器格式（MP4/MKV/FLV）与编解码器（H.264/AAC）
- 使用ffprobe分析媒体文件
- 理解封装与解封装的基本原理！🚀